In [50]:
# 2. Explore the dataset
import os 
from PIL import Image

In [51]:
# Define the directory where the dataset is
base_dir = "CCSN_v2"

# Load the name of all the folders 
class_folders = [folder for folder in os.listdir(base_dir)]

# Load the images 
images_dict = {}
for class_folder in class_folders:
    class_path = os.path.join(base_dir, class_folder)
    image_files = [image_file for image_file in os.listdir(class_path)]

    images_dict[class_folder] = []
    for image_file in image_files:
        image_path = os.path.join(class_path, image_file)
        with Image.open(image_path) as img:
            images_dict[class_folder].append(img.copy())

# Explore numbers on my dataset
print(f"We have a total of: {len(images_dict)} classes (also called labels)")
for key, value in images_dict.items():
    print(f"    - We have {len(value)} of the class '{key}'")
    # Print all the sizes that we have in each class
    size_count_dict = {}
    for image in value:
        size = image.size
        # Get the value (in this case the number of repetitions of the key)
        # size (tuple with the size of the image) and change it for its value
        # plus one. If there were no repetitions yet, get 0 and add 1
        size_count_dict[size] = size_count_dict.get(size, 0) +1
    for size, count in size_count_dict.items():
        print(f"    {count} image(s) of size {size}")


We have a total of: 11 classes (also called labels)
    - We have 221 of the class 'Ac'
    221 image(s) of size (400, 400)
    - We have 188 of the class 'As'
    188 image(s) of size (400, 400)
    - We have 242 of the class 'Cb'
    242 image(s) of size (400, 400)
    - We have 268 of the class 'Cc'
    178 image(s) of size (400, 400)
    90 image(s) of size (256, 256)
    - We have 139 of the class 'Ci'
    139 image(s) of size (400, 400)
    - We have 287 of the class 'Cs'
    240 image(s) of size (400, 400)
    47 image(s) of size (256, 256)
    - We have 200 of the class 'Ct'
    200 image(s) of size (400, 400)
    - We have 182 of the class 'Cu'
    182 image(s) of size (400, 400)
    - We have 274 of the class 'Ns'
    200 image(s) of size (400, 400)
    74 image(s) of size (256, 256)
    - We have 340 of the class 'Sc'
    340 image(s) of size (400, 400)
    - We have 202 of the class 'St'
    202 image(s) of size (400, 400)


In [52]:
# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from tensorflow.keras.utils import to_categorical 
import numpy as np

# Keep only the 400x400 images
for class_folder in images_dict:
    images_dict[class_folder] = [img for img in images_dict[class_folder] if img.size == (400,400)]

# Split our dataset
test_dict = {}
train_val_dict = {}
for class_name, images in images_dict.items():
    train_val_images, test_images = train_test_split(images, test_size=0.2)
    train_val_dict[class_name] = train_val_images
    test_dict[class_name] = test_images

train_dict = {}
val_dict = {}
for class_name, images in train_val_dict.items():
    train_images, val_images = train_test_split(images, test_size = 0.2)
    val_dict[class_name] = val_images
    train_dict[class_name] = train_images

# Split features and labels 
x_train = []
y_train = []
x_val = []
y_val = []
x_test = []
y_test = []

for class_name in train_dict:
    x_train.extend(train_dict[class_name])
    y_train.extend([class_name]*len(train_dict[class_name]))

for class_name in val_dict:    
    x_val.extend(val_dict[class_name])
    y_val.extend([class_name]*len(val_dict[class_name]))
    
for class_name in test_dict:    
    x_test.extend(test_dict[class_name])
    y_test.extend([class_name]*len(test_dict[class_name]))

# Convert to np.array
X_train = [np.array(img, dtype= "float32") for img in x_train]
X_val = [np.array(img, dtype= "float32") for img in x_val]
X_test = [np.array(img, dtype= "float32") for img in x_test]

# Normalize
max_value = np.max(X_train)
X_train = [img/max_value for img in X_train]
X_val = [img/max_value for img in X_val]
X_test = [img/max_value for img in X_test]


# Convert the list of np.arrays to a np.array
X_train = np.stack(X_train)
X_val = np.stack(X_val)
X_test = np.stack(X_test)


# Convert the labels with encoding
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)   # Only fit_transform the first one
y_val = label_encoder.transform(y_val)
y_test = label_encoder.transform(y_test)

# One-hot enconding/categorical
num_classes = len(label_encoder.classes_)
y_train = to_categorical(y_train, num_classes=num_classes)
y_val = to_categorical(y_val, num_classes=num_classes)
y_test = to_categorical(y_test, num_classes=num_classes)

In [53]:
# Sometimes this won't be necessary or ideal

# Reduce to greyscale
X_train = np.mean(X_train, axis=3, keepdims=True)
X_val = np.mean(X_val, axis=3, keepdims=True)
X_test = np.mean(X_test, axis=3, keepdims=True)

# Reduce the number of features (img size)
def avg_pooling(X,pool_size = 2):
    num_samples, height, width, channels = X.shape
    new_height, new_width = height//pool_size, width//pool_size
    
    X_reshape = X.reshape(num_samples, new_height, pool_size, new_width, pool_size, channels)

    X_pooled = np.mean(X_reshape, axis=(2,4), keepdims=True)
    
    return X_pooled.reshape(num_samples, new_height, new_width, channels)

# Apply to all datasets
X_train = avg_pooling(X_train)
X_val = avg_pooling(X_val)
X_test = avg_pooling(X_test)


In [54]:
# Data augmentation
X_train_flipped = np.flip(X_train, axis=2)  # flip horizontally
X_train = np.concatenate([X_train, X_train_flipped], axis=0)
y_train = np.concatenate([y_train, y_train], axis=0)

In [55]:
# Define our model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Conv2D, BatchNormalization, MaxPooling2D, Dropout, Activation
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2


def build_model(input_shape, num_classes):
    
    """
    Build classifier model
    
    This model builds a classifier for the images tha we pass of the clouds from the dataverse dataset.
    
    Args: 
        input_shape(int): shape of images 
        num_classes(int): number of classes
    
    Returns:
        model (tensorflow.keras.model): my model
    """
    
    def conv_block(x, filters):
        """ 
        Convolutional block
        
        This block will contain my convolutional layers for the example of the clouds,
        since we already now regularization, I will use regularization here as well.
        
        Args:
            x (tensor): Tensorflow tensor
            filters (int): filters in my layers
        
        Returns: 
            x (tensor): Tensorflow tensor
        """
        
        x = Conv2D(filters, [3,3], padding = 'same', activation="relu")(x)
        x = BatchNormalization()(x)
        x = Conv2D(filters, [3,3], padding = 'same', activation="relu")(x)
        x = BatchNormalization()(x)
        x = MaxPooling2D(pool_size = (2,2))(x)
        x = Dropout(0.3 if filters >32 else 0.1)(x)
        return x

    def mlp_block(x, units, last):
        """ 
        Create the MLP block.
        
        This is the block that classifies the images from the preprocessing done by the 
        convolutional layers to transform the dog and cat to something more understandable for the mlp.
        
        Args:
            x (tensor): Tensorflow tensor
            units (int): number of nuerons
            last (bool): Tells if we are building the last block
            
        Returns:
            x (tensor): Tensorflow tensor
        """
        
        x = Dense(units, activation="relu",kernel_regularizer=l2(0.01))(x)
        x = BatchNormalization()(x)
        x = Activation("relu")(x)
        
        if not last:
            x = Dropout(0.1)(x)
            
        return x
        
        
    inputs = Input(shape=input_shape)
    x = conv_block(inputs,64)
    # x = conv_block(x,32)
    # x = conv_block(x,32)
    x = conv_block(x,64)
    
    x = GlobalAveragePooling2D()(x) #Flatten() in an alternative way
    
    x = mlp_block(x,512,False)
    # x = mlp_block(x,128,False)
    x = mlp_block(x,32,True)
    
    outputs = Dense(num_classes, activation="softmax")(x)
    
    model = Model(inputs= inputs, outputs= outputs)
    
    return model

In [ ]:
# Training
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

patience = 10

input_shape = X_train.shape[1:]

class_dim = y_train.shape[1]

model = build_model(input_shape, class_dim)
model.compile(optimizer = "adam", loss ="categorical_crossentropy", metrics = ["accuracy"])


reduce_lr = ReduceLROnPlateau(monitor= "val_loss", factor=0.5, patience=5, min_lr = 1e-5, verbose = 1)
early_stopping = EarlyStopping(monitor= "val_loss", patience= patience, restore_best_weights= True)
history = model.fit(
    x=X_train, y = y_train,
    validation_data = (X_val, y_val),
    epochs = 1000,
    batch_size = 8,
    callbacks=[reduce_lr, early_stopping]
)


Epoch 1/1000
 29/372 ━━━━━━━━━━━━━━━━━━━━ 7:57 1s/step - accuracy: 0.0959 - loss: 4.4705

In [ ]:
# Evaluation
import matplotlib.pyplot as plt

# Training and validation losses
plt.figure(figsize= (10,6))
plt.plot(history.history['loss'], label = "Training Loss")
plt.plot(history.history['val_loss'], label = "Validation Loss")
# Add a vertical line where the early stopping was triggered
stopped_epoch = len(history.history('loss'))-patience
plt.axvline(x=stopped_epoch, color='r', linestyle="-", label="Early Stopping (Best Epoch)")
plt.title("Training and validation losses")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred,axis=1)
y_true_classes = np.argmax(y_test, axis=1)

from sklearn.metrics import confusion_matrix
import seaborn as sns
# Confusion matrix
conf_matrix = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(10,8))
sns.heatmap(
    conf_matrix,
    annot=True,          # Muestra los números en las celdas
    fmt='d',             # Formato entero
    cmap='Blues',        # Paleta de color
    xticklabels=label_encoder.classes_,  # nombres de clases
    yticklabels=label_encoder.classes_
)
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()